In [1]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import tqdm
from tqdm.auto import tqdm

/home/xerneas/jupyter-env/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
/home/xerneas/jupyter-env/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

random.shuffle(words)
print(words[:8])
len(words)

['desi', 'kona', 'santo', 'hazaiah', 'yael', 'gita', 'fielder', 'teana']


32033

In [3]:
if (device := torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")):
    print(f"Using device: {device}")

# block_size = 16
epochs = 400
lr = 0.001

Using device: cuda


In [4]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [11]:
def build_dataset(words):
    max_len = max(len(w) for w in words) + 1
    X, Y, M = [], [], []
    for w in words:
        toks = [stoi(c) for c in w] + [0]
        name_len = len(w) + 1
        x = [0] + toks[:-1]
        y = toks
        pad = max_len - len(x)
        x = x + [0] * pad
        y = y + [0] * pad
        m = [1] * name_len + [0] * (max_len - name_len)
        X.append(x)
        Y.append(y)
        M.append(m)
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    M = torch.tensor(M, dtype=torch.float)
    return X, Y, M

In [12]:
X, Y, M = build_dataset(words)
n = X.shape[0]
Xtr, Ytr, Mtr = X[:int(n*0.9)], Y[:int(n*0.9)], M[:int(n*0.9)]
Xdev, Ydev, Mdev = X[int(n*0.9):], Y[int(n*0.9):], M[int(n*0.9):]
Xtr, Ytr, Mtr = [x.to(device) for x in (Xtr, Ytr, Mtr)]
Xdev, Ydev, Mdev = [x.to(device) for x in (Xdev, Ydev, Mdev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)
print(Mdev)

torch.Size([32033, 16]) torch.Size([32033, 16])
torch.Size([28829, 16]) torch.Size([28829, 16])
torch.Size([3204, 16]) torch.Size([3204, 16])
tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.]], device='cuda:0')


In [13]:
trainds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu(), Mtr.cpu())
traindl = DataLoader(
    dataset=trainds,
    batch_size=512,
    pin_memory=False,
    shuffle=True,
    num_workers=16,
    prefetch_factor=16
)

In [14]:
class GRUCell(torch.nn.Module):
    def __init__(self, emb_size, hid_size):
        super().__init__()
        self.update = nn.Linear(emb_size + hid_size, hid_size)
        self.reset = nn.Linear(emb_size + hid_size, hid_size)
        self.candidate = nn.Linear(emb_size + hid_size, hid_size)
    def gate(self, x, h, layer, activation):
        xh = torch.cat([x, h], dim=1)
        return activation(layer(xh))
    def forward(self, x, h):
        z = self.gate(x, h, self.update, torch.sigmoid) # how much new memory to use
        r = self.gate(x, h, self.reset, torch.sigmoid) # how much old affects new
        h_tilde = self.gate(x, r * h, self.candidate, torch.tanh) # new memory guess

        h = (1-z) * h + z * h_tilde
        return h

In [ ]:
class GRU(torch.nn.Module):
    def __init__(self, vocab_size, emb_size, hid_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size) #embedding ts
        self.cell = GRUCell(emb_size, hid_size)
        self.head = nn.Linear(hid_size, vocab_size)
        self.n_hidden = hid_size # get the probs
    def forward(self, X, h=None):
        B,T = X.shape

        if h is None:
            h = torch.zeros(B, self.n_hidden, device=X.device)

        emb = self.embedding(X)  #(B, T, emb_size)
        logits = []

        for t in range(T):
            x = emb[:, t, :]
            h = self.cell(x, h)
            logits_t = self.head(h)
            logits.append(logits_t)

        logits = torch.stack(logits, dim=1)  #(B, T, vocab_size)
        return logits, h


In [16]:
model = GRU(vocab_size=27, emb_size=10, hid_size=100).to(device)
# model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma = 0.99995)

In [17]:
model.train()
pbar = tqdm(range(epochs), desc="train")
for e in pbar:
    for Xb, Yb, Mb in traindl:
        Xb, Yb, Mb = Xb.to(device), Yb.to(device), Mb.to(device)
        logits, _ = model(Xb)
        per_pos = F.cross_entropy(logits.reshape(-1, 27), Yb.reshape(-1), reduction='none')
        loss = (per_pos * Mb.reshape(-1)).sum() / Mb.sum()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    # if e % 10 == 0:
    #     print(e, loss.item(), scheduler.get_last_lr()[0])

train: 100%|██████████| 400/400 [02:45<00:00,  2.42it/s, loss=1.8254, lr=3.20e-04]


In [33]:
model.eval()
with torch.no_grad():
    logits, _ = model(Xdev)
    per_pos = F.cross_entropy(logits.reshape(-1, 27), Ydev.reshape(-1), reduction='none')
    dev_loss = (per_pos * Mdev.reshape(-1)).sum() / Mdev.sum()
    print('dev:', dev_loss.item())

dev: 2.0282278060913086


In [39]:
model.eval()
for _ in range(10):
    out = []
    h = None #blank hidden space for now
    x = torch.tensor([[0]], device=device)
    max_steps = 30 #hard cutoff

    for i in range(max_steps):
        with torch.no_grad():
            logits, h = model(x, h)
        probs = F.softmax(logits[0, -1], dim=0)
        ix = torch.multinomial(probs, num_samples=1).item()
        if ix == 0:
            break
        out.append(itos(ix))
        x = torch.tensor([[ix]], device=device)

    print(''.join(out))

sebant
delubah
iniyan
kugen
banling
brendyn
simeri
allani
madaina
fazir
